# BlindNav — Real-Time Assistive AI for Visually Impaired Navigation

**Pipeline:** BLIP-2 (scene caption) + YOLOv8-nano (object detection) + MiDaS (depth) → Priority Engine → TTS narration

**Testing:** Run on Colab with any image URL or local file path.

**Production:** Android app posts a camera frame to the FastAPI server (see Cell 7), gets back JSON with narration text and speaks it using Android TTS.

No hardcoded paths. No Google Drive dependencies.

## 1. Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "transformers>=4.42.0 accelerate>=0.30.0",
    "ultralytics>=8.2.50",
    "opencv-python pillow",
    "timm>=1.0.0",
    "gtts",
    "fastapi uvicorn[standard] python-multipart",
    "nest-asyncio",
    "requests tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkg.split())

print("All packages installed.")

## 2. Imports & Device

In [ ]:
import base64
import io
import time
import warnings
from collections import deque
from dataclasses import dataclass
from enum import IntEnum

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import Blip2ForConditionalGeneration, Blip2Processor
from ultralytics import YOLO

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("medium")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Load Models

Models are cached after first download. Run this cell once per Colab session.

In [ ]:
# --- BLIP-2 ---
BLIP2_MODEL_ID = "Salesforce/blip2-flan-t5-xl"
print("Loading BLIP-2...")
blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL_ID)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
blip2_model.eval()
print("BLIP-2 ready.")

# --- YOLOv8-nano ---
print("Loading YOLOv8-nano...")
yolo_model = YOLO("yolov8n.pt")
yolo_model.to(DEVICE)
print("YOLOv8-nano ready.")

# --- MiDaS small ---
print("Loading MiDaS...")
midas_model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)
midas_model = midas_model.to(DEVICE)
midas_model.eval()
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True)
midas_transform = midas_transforms.small_transform
print("MiDaS ready.")

## 4. Core Pipeline Functions

In [ ]:
# -----------------------------------------------------------------------
# Image loading
# Accepts: file path, HTTP URL, base64 data-URI, raw bytes, PIL Image.
# Same function used for Colab test URLs and Android POST payloads.
# -----------------------------------------------------------------------

import requests as _requests


def load_image(source) -> Image.Image:
    if isinstance(source, Image.Image):
        return source.convert("RGB")
    if isinstance(source, bytes):
        return Image.open(io.BytesIO(source)).convert("RGB")
    if isinstance(source, str):
        if source.startswith("data:image"):
            b64 = source.split(",", 1)[1]
            return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
        if source.startswith("http"):
            resp = _requests.get(source, timeout=15)
            resp.raise_for_status()
            return Image.open(io.BytesIO(resp.content)).convert("RGB")
        # try raw base64
        if not source.startswith(("/", ".", "~")):
            try:
                return Image.open(io.BytesIO(base64.b64decode(source))).convert("RGB")
            except Exception:
                pass
        return Image.open(source).convert("RGB")
    raise TypeError(f"Unsupported image source type: {type(source)}")

In [ ]:
# -----------------------------------------------------------------------
# BLIP-2 Scene Caption
# -----------------------------------------------------------------------

NAV_PROMPT = "Describe this scene concisely for a visually impaired person navigating:"


def generate_caption(
    image: Image.Image,
    prompt: str = NAV_PROMPT,
    max_new_tokens: int = 60,
) -> tuple:
    """Returns (caption_str, latency_seconds)."""
    t0 = time.perf_counter()
    inputs = blip2_processor(images=image, text=prompt, return_tensors="pt")
    inputs = {
        k: v.to(DEVICE, torch.float16 if v.is_floating_point() else v.dtype)
        for k, v in inputs.items()
    }
    with torch.no_grad():
        output_ids = blip2_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            repetition_penalty=2.5,
            length_penalty=1.5,
        )
    caption = blip2_processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    return caption, time.perf_counter() - t0

In [ ]:
# -----------------------------------------------------------------------
# YOLOv8 Object Detection
# -----------------------------------------------------------------------

YOLO_CONF = 0.35

DANGER_CLASSES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "fire hydrant", "stop sign", "traffic light",
    "chair", "bench", "backpack", "suitcase", "dog", "cat",
}


@dataclass
class Detection:
    label: str
    confidence: float
    box_xyxy: tuple        # normalised (x1, y1, x2, y2) in [0, 1]
    relative_area: float


def detect_objects(image: Image.Image) -> tuple:
    """Returns (list[Detection], latency_seconds)."""
    t0 = time.perf_counter()
    results = yolo_model.predict(source=image, conf=YOLO_CONF, verbose=False)
    w, h = image.size
    detections = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            nx1, ny1, nx2, ny2 = x1 / w, y1 / h, x2 / w, y2 / h
            area = (nx2 - nx1) * (ny2 - ny1)
            detections.append(Detection(
                label=r.names[int(box.cls[0])],
                confidence=float(box.conf[0]),
                box_xyxy=(nx1, ny1, nx2, ny2),
                relative_area=area,
            ))
    return detections, time.perf_counter() - t0

In [ ]:
# -----------------------------------------------------------------------
# MiDaS Depth Estimation
# Inverse depth: higher value = object is CLOSER. Normalised to [0, 1].
# -----------------------------------------------------------------------


def estimate_depth(image: Image.Image) -> tuple:
    """Returns (depth_map[H, W] float32, latency_seconds)."""
    t0 = time.perf_counter()
    img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    tensor = midas_transform(img_bgr).to(DEVICE)
    with torch.no_grad():
        pred = midas_model(tensor)
        pred = F.interpolate(
            pred.unsqueeze(1),
            size=(image.height, image.width),
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    depth = pred.cpu().numpy().astype(np.float32)
    d_min, d_max = depth.min(), depth.max()
    if d_max - d_min > 1e-6:
        depth = (depth - d_min) / (d_max - d_min)
    return depth, time.perf_counter() - t0


def depth_scores_for_detections(detections: list, depth_map: np.ndarray) -> list:
    """Mean inverse-depth score per detection bounding box."""
    H, W = depth_map.shape
    scores = []
    for det in detections:
        x1, y1, x2, y2 = det.box_xyxy
        roi = depth_map[
            max(0, int(y1 * H)): min(H, int(y2 * H)),
            max(0, int(x1 * W)): min(W, int(x2 * W)),
        ]
        scores.append(float(roi.mean()) if roi.size > 0 else 0.0)
    return scores

In [ ]:
# -----------------------------------------------------------------------
# Priority Engine
# Ranks detections danger-first so TTS always speaks the most urgent
# obstacle first. This is what makes BlindNav research-level.
# -----------------------------------------------------------------------

class DistanceClass(IntEnum):
    CRITICAL = 0   # < ~1.5 m
    CAUTION  = 1   # 1.5 – 3 m
    CLEAR    = 2   # > 3 m


CRITICAL_THRESH = 0.55
CAUTION_THRESH  = 0.35

DISTANCE_LABELS = {
    DistanceClass.CRITICAL: "less than 1.5 metres ahead",
    DistanceClass.CAUTION:  "about 1.5 to 3 metres ahead",
    DistanceClass.CLEAR:    "more than 3 metres ahead",
}


@dataclass
class RankedDetection:
    detection: Detection
    depth_score: float
    distance_class: DistanceClass
    priority_score: float


def rank_detections(detections: list, depth_scores: list) -> list:
    """Returns list[RankedDetection] sorted most-urgent first."""
    ranked = []
    for det, score in zip(detections, depth_scores):
        if score >= CRITICAL_THRESH:
            dc = DistanceClass.CRITICAL
        elif score >= CAUTION_THRESH:
            dc = DistanceClass.CAUTION
        else:
            dc = DistanceClass.CLEAR

        is_danger = det.label.lower() in DANGER_CLASSES
        priority = int(dc) * 10.0 - (0.2 if is_danger else 0.0) - det.relative_area
        ranked.append(RankedDetection(
            detection=det,
            depth_score=score,
            distance_class=dc,
            priority_score=priority,
        ))

    ranked.sort(key=lambda x: x.priority_score)
    return ranked

In [ ]:
class TemporalSmoother:
    def __init__(self, window: int = 5):
        self._history: deque = deque(maxlen=window)

    def _key(self, rd: RankedDetection) -> tuple:
        return (rd.detection.label, rd.distance_class)

    def filter(self, ranked: list) -> list:
        known = set()
        for frame in self._history:
            known.update(frame)
        current_keys = {self._key(rd) for rd in ranked}
        new_items = [rd for rd in ranked if self._key(rd) not in known]
        self._history.append(current_keys)
        return new_items


smoother = TemporalSmoother(window=5)

In [ ]:
# -----------------------------------------------------------------------
# Narration Builder
# Produces the final spoken sentence. CRITICAL objects always first.
# -----------------------------------------------------------------------


def build_narration(caption: str, ranked: list, max_obstacles: int = 3) -> str:
    parts = []
    for rd in ranked[:max_obstacles]:
        label = rd.detection.label.capitalize()
        dist  = DISTANCE_LABELS[rd.distance_class]
        parts.append(f"{label} {dist}.")
    parts.append(caption.rstrip(".") + ".")
    return " ".join(parts)

In [ ]:
# -----------------------------------------------------------------------
# TTS — gTTS plays audio inline in Colab.
# On Android the app receives the narration text and calls Android TTS
# natively (no audio bytes sent over the network).
# -----------------------------------------------------------------------

from gtts import gTTS
from IPython.display import Audio, display


def speak(text: str) -> None:
    """Synthesise and play narration audio inline in the notebook."""
    try:
        buf = io.BytesIO()
        gTTS(text=text, lang="en", slow=False).write_to_fp(buf)
        buf.seek(0)
        display(Audio(buf.read(), rate=24000, autoplay=True))
    except Exception as err:
        print(f"[TTS fallback] {text}  ({err})")


print("TTS ready.")

## 5. End-to-End Pipeline

`run_pipeline(source)` — one call, any image source.

In [ ]:
def run_pipeline(
    source,
    speak_output: bool = False,
    use_temporal_filter: bool = True,
    max_obstacles: int = 3,
    caption_prompt: str = NAV_PROMPT,
) -> dict:
    """
    Full BlindNav inference pipeline.

    Parameters
    ----------
    source              : image (path / URL / bytes / base64 / PIL)
    speak_output        : play TTS audio in the notebook
    use_temporal_filter : suppress repeated detections across frames
    max_obstacles       : max obstacles included in narration
    caption_prompt      : BLIP-2 instruction prompt

    Returns
    -------
    dict with keys: narration, caption, detections, latency
    """
    t_total = time.perf_counter()

    image = load_image(source)

    caption, t_cap = generate_caption(image, prompt=caption_prompt)
    detections, t_det = detect_objects(image)
    depth_map, t_dep = estimate_depth(image)
    depth_scores = depth_scores_for_detections(detections, depth_map)

    ranked = rank_detections(detections, depth_scores)
    ranked_filtered = smoother.filter(ranked) if use_temporal_filter else ranked

    narration = build_narration(caption, ranked_filtered, max_obstacles=max_obstacles)

    if speak_output:
        speak(narration)

    total_latency = time.perf_counter() - t_total

    return {
        "narration": narration,
        "caption": caption,
        "detections": [
            {
                "label": rd.detection.label,
                "confidence": round(rd.detection.confidence, 3),
                "box_xyxy": rd.detection.box_xyxy,
                "distance_class": rd.distance_class.name,
                "distance_label": DISTANCE_LABELS[rd.distance_class],
                "depth_score": round(rd.depth_score, 3),
            }
            for rd in ranked
        ],
        "latency": {
            "caption_s": round(t_cap, 3),
            "detection_s": round(t_det, 3),
            "depth_s": round(t_dep, 3),
            "total_s": round(total_latency, 3),
        },
        # keep depth_map in result for visualisation (not serialised in API)
        "_depth_map": depth_map,
        "_image": image,
    }

## 6. Full Pipeline Test

Uses a busy street photo (people + cars + bikes). Runs all models, prints results,
draws YOLO bounding boxes colour-coded by danger zone, shows the MiDaS depth heatmap, and plays the narration audio.

Change `TEST_IMAGE` to any URL or local path.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# -----------------------------------------------------------------------
# Street scene: people, cars, bikes — exercises all three models.
# Swap for any URL or local path (e.g. "/content/my_photo.jpg").
# -----------------------------------------------------------------------
TEST_IMAGE = "/content/image.jpg"


# Run the full pipeline with audio output
result = run_pipeline(TEST_IMAGE, speak_output=True, use_temporal_filter=False)

image     = result["_image"]
depth_map = result["_depth_map"]
w, h      = image.size

# -----------------------------------------------------------------------
# Print results
# -----------------------------------------------------------------------
print("=" * 65)
print("NARRATION:", result["narration"])
print("-" * 65)
print("CAPTION  :", result["caption"])
print("-" * 65)
print(f"{'LABEL':<20} {'CLASS':<10} {'DEPTH':>6}  {'CONF':>5}")
print("-" * 65)
for d in result["detections"]:
    print(f"{d['label']:<20} {d['distance_class']:<10} {d['depth_score']:>6.2f}  {d['confidence']:>5.2f}")
print("-" * 65)
for k, v in result["latency"].items():
    print(f"  {k}: {v:.3f} s")
print("=" * 65)


COLOUR_MAP = {
    "CRITICAL": (220,  30,  30),
    "CAUTION" : (240, 160,  20),
    "CLEAR"   : ( 40, 200,  80),
}

annotated = np.array(image).copy()
for d in result["detections"]:
    x1n, y1n, x2n, y2n = d["box_xyxy"]
    colour = COLOUR_MAP.get(d["distance_class"], (128, 128, 128))
    pt1 = (int(x1n * w), int(y1n * h))
    pt2 = (int(x2n * w), int(y2n * h))
    cv2.rectangle(annotated, pt1, pt2, colour, 2)
    label_text = f"{d['label']} {d['distance_class'][0]}"
    cv2.putText(annotated, label_text,
                (pt1[0], max(10, pt1[1] - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2)


depth_vis = (depth_map * 255).astype(np.uint8)
depth_colour = cv2.applyColorMap(depth_vis, cv2.COLORMAP_MAGMA)
depth_colour = cv2.cvtColor(depth_colour, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(np.array(image))
axes[0].set_title("Original", fontsize=13)
axes[0].axis("off")

axes[1].imshow(annotated)
axes[1].set_title("YOLO Detections", fontsize=13)
axes[1].axis("off")
legend_patches = [
    mpatches.Patch(color=(220/255, 30/255,  30/255), label="CRITICAL  (<1.5 m)"),
    mpatches.Patch(color=(240/255, 160/255, 20/255), label="CAUTION   (1.5-3 m)"),
    mpatches.Patch(color=( 40/255, 200/255, 80/255), label="CLEAR     (>3 m)"),
]
axes[1].legend(handles=legend_patches, loc="lower left", fontsize=9)

axes[2].imshow(depth_colour)
axes[2].set_title("MiDaS Depth  (bright = closer)", fontsize=13)
axes[2].axis("off")

plt.suptitle(
    f"BlindNav   |   total latency: {result['latency']['total_s']:.2f} s",
    fontsize=15, y=1.01
)
plt.tight_layout()
plt.show()

## 7. Latency Benchmark

Target: end-to-end < 2 seconds. Run after models are loaded (avoids cold-start noise).

In [ ]:
N_RUNS = 5
BENCH_IMAGE = TEST_IMAGE

timings = []
for i in range(N_RUNS):
    r = run_pipeline(BENCH_IMAGE, speak_output=False, use_temporal_filter=False)
    timings.append(r["latency"])
    print(
        f"Run {i + 1}: total={r['latency']['total_s']:.3f}s  "
        f"cap={r['latency']['caption_s']:.3f}s  "
        f"det={r['latency']['detection_s']:.3f}s  "
        f"dep={r['latency']['depth_s']:.3f}s"
    )

avg_total = sum(t["total_s"] for t in timings) / N_RUNS
print(f"\nAverage over {N_RUNS} runs: {avg_total:.3f} s")
print("Target <2s: PASS" if avg_total < 2.0 else "Target <2s: FAIL — see quantization cell below.")

## 8. Optional: INT8 Quantization

Only run if benchmark fails. Quantizes MiDaS linear layers to INT8. BLIP-2 is already FP16.

In [ ]:
midas_int8 = torch.quantization.quantize_dynamic(
    midas_model.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)
midas_model = midas_int8   # swap global reference
print("MiDaS quantized to INT8.")

r = run_pipeline(BENCH_IMAGE, speak_output=False, use_temporal_filter=False)
print(f"Post-quantization latency: {r['latency']['total_s']:.3f} s")